# Figure 6: cell-cell contacts

Each panel is one immune cell type against one breast cancer cell type, with or without an antibody
bridging them: the trans pairs that can form across the contact, the gap each holds the membranes at,
each pair's share of the contact with every protein's abundance shared among its partners by mass
action, which proteins on each side are excluded from the contact by height, and the cis pairs each
side carries into it. `data/curated/contact_panels.csv` names the panels; the computation is
`surfaceomeTopography.contact`, once for all of them. Heights are extended estimates, upper bounds
for flexible proteins, so gaps and exclusion are overstated.

In [ ]:
from surfaceome_config import resolve, out_path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()   #the panels' figure style
%matplotlib inline

from surfaceomeTopography import (percent_expression, expressed_complexes, surface_units, contact_pairs,
                                  contact_exclusion, cis_pairs_at_contact, contact_layout,
                                  plot_contact_bullseye, plot_surfaces, plot_clearance, plot_contact_matrix)

In [ ]:
#the panels: immune cell sample (E-PROT-1), cancer sample (E-PROT-27), the antibody's antigen if one
#bridges the contact, and the cancer protein added to its surface even when in a complex (the antibody's target)
panels = pd.read_csv(resolve('curated/contact_panels.csv'), keep_default_na = False)
panels

In [ ]:
#the surfaceome's names (one row per accession), the two proteomes, the interaction and height tables
names = pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'), sep = '\t')
names_cleanup = pd.read_csv(resolve('inputs/mapping/uniprot_mapping_cleanup.tab'), index_col = None, sep = '\t')
surfaceome = pd.concat([names, names_cleanup]).drop_duplicates(subset = ['ID link'])

expression_immune = pd.read_csv(resolve('inputs/expression/E-PROT-1-query-results.tsv'), sep = '\t', header = 4)
expression_cancer = pd.read_csv(resolve('inputs/expression/E-PROT-27-query-results.tsv'), sep = '\t', header = 4)

#partner heights are kept: an antibody bridge's gap is computed from the antigen's
interactions = pd.read_csv(resolve('database/interaction_heights.csv'))
interactions = interactions.drop(columns = ['ID link_prot1', 'ID_prot1', 'ID link_prot2', 'ID_prot2'])

heights = pd.read_csv(resolve('database/height_estimates.csv'))
heights = heights.rename(columns = {'ID link_first': 'ID link', 'Entry name_first': 'Entry name',
                                    'dom_len_count_domain': 'domain counts'})
heights = heights.drop(columns = ['max_dim_first', 'min_dim_first', 'seq_len_first', 'source_last'])

from surfaceomeTopography.interactions import with_curated_complexes
complexes = with_curated_complexes(pd.read_csv(resolve('inputs/cellphonedb/complex_curated.csv')),
                                   pd.read_csv(resolve('curated/curated_complexes.csv')))
cis_pairs = pd.read_csv(resolve('curated/cis_pairs.csv'))

#where each antibody's epitope stands on its antigen, by the antigen's accession
epitope_heights = pd.read_csv(resolve('database/antibody_epitope_heights.csv')).set_index('accession')['epitope_height']

## The two proteomes as surfaces

Surface proteins with a height and an expression value, one row per protein and sample, each with
its share of its sample's summed abundance; then each sample's complexes (CellphoneDB) that are
expressed, and the units a surface is made of.

In [ ]:
ids = ['ID link', 'Entry', 'Entry name', 'Status', 'Protein names', 'Gene names', 'Organism', 'Length',
       'Gene names  (primary )', 'Gene ID', 'Gene Name', 'ID', 'methods_<lambda>', 'total_height']

def surface_expression(expression):
    #the samples are every column after the two identifier columns
    surf = pd.merge(surfaceome, expression, left_on = 'Gene names  (primary )', right_on = 'Gene Name', how = 'inner')
    surf = pd.merge(surf, heights, on = ['ID link', 'Entry name'])
    long = pd.melt(surf, id_vars = ids, value_vars = list(expression.columns[2:]),
                   var_name = 'sample', value_name = 'expression')
    long['percent_expression'] = percent_expression(long)
    return long

expression_long_immune = surface_expression(expression_immune)
expression_long_cancer = surface_expression(expression_cancer)
complex_expression_immune = expressed_complexes(expression_long_immune, complexes)
complex_expression_cancer = expressed_complexes(expression_long_cancer, complexes)
expression_long_immune.groupby('sample')['ID link'].count()

## The panels

For each panel: the two surfaces (`tables/F6_<panel>/`), the pairs across the contact with their gaps
and shares, the bullseye and the gap distribution (Figure 6), the exclusion and the cis pairs on each
side, and the interaction-by-protein clearance matrix with its margins.

In [ ]:
#TrueType text in every PDF, so the figures stay editable
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

summary = {}
for p in panels.itertuples():
    immune = surface_units(expression_long_immune, complex_expression_immune, p.immune_sample)
    cancer = surface_units(expression_long_cancer, complex_expression_cancer, p.cancer_sample,
                           include = [p.cancer_include] if p.cancer_include else [])
    pairs = contact_pairs(immune, cancer, interactions, antibody_target = p.antibody_target or None,
                          epitope_heights = epitope_heights)

    t = f'tables/F6_{p.panel}/{p.panel}_'
    immune.to_csv(out_path(t + 'immune_surface.csv'), index = False)
    cancer.to_csv(out_path(t + 'cancer_surface.csv'), index = False)
    pairs.to_csv(out_path(t + 'pairs.csv'), index = False)

    #Figure 6: the contact's gap bands by the shared-abundance weight, and the gap distribution
    plot_contact_bullseye(pairs).savefig(out_path(f'figures/F6_{p.panel}_bullseye.pdf'), transparent = True, bbox_inches = 'tight')
    plot_surfaces({p.label: pairs}, height = 'interaction_dim', weight = 'interface_shared').savefig(
        out_path(f'figures/F6_{p.panel}_contact_gaps.pdf'), transparent = True)

    #exclusion: a protein about 5 nm taller than a gap is excluded from it; cis pairs on each side
    exclusion = {'immune': contact_exclusion(immune, pairs), 'cancer': contact_exclusion(cancer, pairs)}
    cis = {'immune': cis_pairs_at_contact(immune, cis_pairs, pairs, 'immune'),
           'cancer': cis_pairs_at_contact(cancer, cis_pairs, pairs, 'cancer')}
    for side in ('immune', 'cancer'):
        exclusion[side].to_csv(out_path(t + f'{side}_contact_exclusion.csv'), index = False)
        cis[side].to_csv(out_path(t + f'{side}_cis_pairs.csv'), index = False)

    #the clearance matrix: pairs holding 0.1% or more of the contact against the immune units above 1%
    td, units, vs = contact_layout(pairs, immune)
    plot_clearance(vs).savefig(out_path(f'figures/F6_{p.panel}_interactions_vs_expression.svg'), format = 'svg')
    plot_contact_matrix(vs, units, td).savefig(out_path(f'figures/F6_{p.panel}_interactions_vs_proteins.pdf'),
                                               format = 'pdf', transparent = True)
    plt.close('all')

    summary[p.panel] = {'immune units': len(immune), 'cancer units': len(cancer), 'pairs': len(pairs),
                        'antibody bridges': int((pairs['_merge'] == 'FcR').sum()),
                        'share of the contact in pairs above 0.1%': round(td['interface_shared'].sum(), 1),
                        **{f'{side} surface excluded, %': round(np.average(exclusion[side].excluded, weights = exclusion[side].percent_expression), 1)
                           for side in ('immune', 'cancer')}}
pd.DataFrame(summary).T